# PRACTICAL 06: PySpark DataFrame Transformations: Selection, Filtering & Columns
**[UNIT 2: Big Data Processing using PySpark]**


### Problem Statement:
Write a PySpark program to execute DataFrame transformations: select specific columns using .select(), filter rows using multi-condition boolean logic (e.g., Salary > 60000 and City == 'Surat') using .filter(), create a computed column using .withColumn(), and rename a column using .withColumnRenamed().

### Expected Reference Output:
```text
--- 1. SELECT NAME AND SALARY ---
+-----+------+
| Name|Salary|
+-----+------+
|Aarav| 65000|
| Diya| 45000|
|Rohan| 72000|
...
+-----+------+

--- 2. FILTER (Dept == 'IT' AND Salary > 70000) ---
+---+-----+----+------+-----+
| ID| Name|Dept|Salary| City|
+---+-----+----+------+-----+
|  3|Rohan|  IT| 72000|Surat|
|  5|Kabir|  IT| 81000|Surat|
+---+-----+----+------+-----+

--- 3. ADD COMPUTED 'Bonus' COLUMN ---
+---+-----+----+------+-----+------+
| ID| Name|Dept|Salary| City| Bonus|
+---+-----+----+------+-----+------+
|  1|Aarav|  IT| 65000|Surat|6500.0|
...
```

In [1]:
import os
import sys
import pyspark

# Configure Spark, Python & Java paths explicitly to prevent gateway launch failures
spark_home = os.path.dirname(pyspark.__file__)
os.environ["SPARK_HOME"] = spark_home
os.environ["SPARK_SCALA_VERSION"] = "2.13"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Resolve and validate JAVA_HOME (clean trailing slashes)
java_candidates = [
    os.environ.get("JAVA_HOME", "").rstrip("/\\"),
    r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot",
    os.path.expanduser(r"~\AppData\Local\Java\jdk-17")
]
for jc in java_candidates:
    if jc and os.path.isfile(os.path.join(jc, "bin", "java.exe")):
        os.environ["JAVA_HOME"] = jc
        os.environ["PATH"] = os.path.join(jc, "bin") + os.pathsep + os.environ.get("PATH", "")
        break

# Resolve Hadoop winutils for Windows support
hadoop_candidates = [
    os.environ.get("HADOOP_HOME", "").rstrip("/\\"),
    os.path.expanduser(r"~\AppData\Local\hadoop")
]
for hc in hadoop_candidates:
    if hc and os.path.isdir(hc):
        os.environ["HADOOP_HOME"] = hc
        os.environ["PATH"] = os.path.join(hc, "bin") + os.pathsep + os.environ.get("PATH", "")
        break

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DS505_Transformations").master("local[*]").getOrCreate()

C:\Users\Prem Patel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# Create sample dataset
data = [
    (1, "Aarav", "IT", 65000, "Surat"),
    (2, "Diya", "HR", 45000, "Navsari"),
    (3, "Rohan", "IT", 72000, "Surat"),
    (4, "Pooja", "Finance", 58000, "Ahmedabad"),
    (5, "Kabir", "IT", 81000, "Surat"),
    (6, "Neha", "Finance", 62000, "Surat")
]
columns = ["ID", "Name", "Dept", "Salary", "City"]
df = spark.createDataFrame(data, columns)

print("--- ORIGINAL DATAFRAME ---")
df.show()

--- ORIGINAL DATAFRAME ---


+---+-----+-------+------+---------+
| ID| Name|   Dept|Salary|     City|
+---+-----+-------+------+---------+
|  1|Aarav|     IT| 65000|    Surat|
|  2| Diya|     HR| 45000|  Navsari|
|  3|Rohan|     IT| 72000|    Surat|
|  4|Pooja|Finance| 58000|Ahmedabad|
|  5|Kabir|     IT| 81000|    Surat|
|  6| Neha|Finance| 62000|    Surat|
+---+-----+-------+------+---------+



In [3]:
# 1. Column Selection
print("--- 1. SELECT NAME AND SALARY ---")
df.select("Name", "Salary").show()

--- 1. SELECT NAME AND SALARY ---


+-----+------+
| Name|Salary|
+-----+------+
|Aarav| 65000|
| Diya| 45000|
|Rohan| 72000|
|Pooja| 58000|
|Kabir| 81000|
| Neha| 62000|
+-----+------+



In [4]:
# 2. Multi-condition Filtering
print("--- 2. FILTER (Dept == 'IT' AND Salary > 70000) ---")
df.filter((col("Dept") == "IT") & (col("Salary") > 70000)).show()

--- 2. FILTER (Dept == 'IT' AND Salary > 70000) ---


+---+-----+----+------+-----+
| ID| Name|Dept|Salary| City|
+---+-----+----+------+-----+
|  3|Rohan|  IT| 72000|Surat|
|  5|Kabir|  IT| 81000|Surat|
+---+-----+----+------+-----+



In [5]:
# 3. Add Computed Column (Bonus = 10% of Salary)
print("--- 3. ADD COMPUTED 'Bonus' COLUMN ---")
df_with_bonus = df.withColumn("Bonus", col("Salary") * 0.10)
df_with_bonus.show()

--- 3. ADD COMPUTED 'Bonus' COLUMN ---


+---+-----+-------+------+---------+------+
| ID| Name|   Dept|Salary|     City| Bonus|
+---+-----+-------+------+---------+------+
|  1|Aarav|     IT| 65000|    Surat|6500.0|
|  2| Diya|     HR| 45000|  Navsari|4500.0|
|  3|Rohan|     IT| 72000|    Surat|7200.0|
|  4|Pooja|Finance| 58000|Ahmedabad|5800.0|
|  5|Kabir|     IT| 81000|    Surat|8100.0|
|  6| Neha|Finance| 62000|    Surat|6200.0|
+---+-----+-------+------+---------+------+



In [6]:
# 4. Rename Column
print("--- 4. RENAME 'Salary' TO 'Base_Salary' ---")
df_renamed = df_with_bonus.withColumnRenamed("Salary", "Base_Salary")
df_renamed.show()

# Stop SparkSession cleanly to release background Java worker processes
spark.stop()

--- 4. RENAME 'Salary' TO 'Base_Salary' ---


+---+-----+-------+-----------+---------+------+
| ID| Name|   Dept|Base_Salary|     City| Bonus|
+---+-----+-------+-----------+---------+------+
|  1|Aarav|     IT|      65000|    Surat|6500.0|
|  2| Diya|     HR|      45000|  Navsari|4500.0|
|  3|Rohan|     IT|      72000|    Surat|7200.0|
|  4|Pooja|Finance|      58000|Ahmedabad|5800.0|
|  5|Kabir|     IT|      81000|    Surat|8100.0|
|  6| Neha|Finance|      62000|    Surat|6200.0|
+---+-----+-------+-----------+---------+------+

